# 👤 4. Human-in-the-Loop Middleware

Human-in-the-loop (HITL) middleware allows an AI agent to **pause execution and request human approval** before performing sensitive operations.

This is useful when an action can have significant real-world consequences and should not happen automatically.

---

## 🧩 How Human-in-the-Loop Works

```text
User Request
    ↓
AI Agent
    ↓
Sensitive Operation?
    ├── No  → ✅ Continue
    └── Yes
          ↓
    ⏸️ Pause Execution
          ↓
    👤 Human Approval
       ├── Approve → ✅ Continue
       └── Reject  → 🚫 Stop

---

## 🎯 When Should You Use HITL?

Human approval is especially useful for:

- 💰 Financial transactions
- 📧 Sending external emails
- 🗑️ Deleting production data
- ⚠️ High-impact operations
- 🔐 Sensitive tool calls

---

## 🛠️ HumanInTheLoopMiddleware

LangChain provides built-in `HumanInTheLoopMiddleware` for adding human approval checkpoints to agent workflows.

The middleware can pause execution **before sensitive tool calls** and wait for a human decision.

## 👤 Human-in-the-Loop Middleware Example

### Import Required Libraries

In [1]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

### Groq model

In [2]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

### Create agent with HITL middleware

In [3]:
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


### Step 1: Invoke — agent will pause before send_email

In [4]:
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Send an email to team@company.com about the Q4 results"
            }
        ]
    },
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='e2bbefe5-d8d0-4200-a345-cb3418753bd2'), AIMessage(content='', additional_kwargs={'reasoning_content': "We need to send email. Use send_email function. Subject? Not specified. We'll craft a brief email about Q4 results. Provide body. Use function.", 'tool_calls': [{'id': 'fc_45febecd-8556-4697-9f45-e6e865084fc9', 'function': {'arguments': '{"body":"Hi Team,\\n\\nI wanted to share a quick overview of our Q4 results:\\n\\n- Revenue increased by 12% year-over-year, reaching $8.5M.\\n- Net profit margin improved to 15%, up from 13% in Q3.\\n- Customer acquisition grew by 8%, with a notable rise in enterprise accounts.\\n- Operational costs were reduced by 5% due to efficiency initiatives.\\n\\nOverall, we are on track to meet our annual targets, and the momentum looks promising heading into the new year.\\

### Step 2: Human reviews and APPROVES

In [5]:
approved_result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=config  # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email has been sent to team@company.com with the subject “Q4 Results Overview.” Let me know if there’s anything else you’d like to add or any other tasks you need help with!


### Step 3: Alternative — Human REJECTS

In [6]:
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete all records from the users table where active=false"
            }
        ]
    },
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "reason": "Too risky, needs DBA review"
                }
            ]
        }
    ),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I’m ready to delete the records you specified, but I want to make sure that’s really what you intend. Would you like me to proceed with deleting **all rows in the `users` table where `active = false`**? Please confirm, and let me know if there are any additional conditions or safeguards you’d like applied.


---

## 💡 Key Idea

> **Human-in-the-loop middleware adds a human approval checkpoint before sensitive agent actions.**

Instead of allowing the agent to perform every action automatically, humans can review and approve important operations.

---

## 📝 Key Takeaways

- `HumanInTheLoopMiddleware` allows agents to **pause for human approval**.
- It is useful for **sensitive and high-impact operations**.
- Common use cases include financial transactions, external emails, and production data deletion.
- HITL middleware can interrupt execution **before sensitive tool calls**.
- A **checkpointer is required** to persist the agent's state while execution is paused.
- HITL provides an additional layer of **safety and control** for AI agents.